In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.append(str(SRC_PATH))

In [ ]:
# set global seaborn style
sns.set_theme(
    context="notebook",
    style="ticks",
    palette="muted",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# Data Cleaning

In [ ]:
df = pd.read_csv("../data/raw/stroke.csv")

In [ ]:
# drop ID column to prevent accidental usage
if "id" in df.columns:
    df = df.drop(columns=["id"])

In [ ]:
from preprocessing import normalize_strings

df = normalize_strings(df)

## Missing Values

In [ ]:
df.isnull().sum()

A coluna `bmi` possui 201 valores ausentes. Existe diferentes estratégias para lidar com valores ausentes, incluindo remoção das observações, imputar novos valores ou deixá-los como estão.

Considerando o número relativamente grande de valores ausentes, é razoável imputar esses valores.

Iremos avaliar:
  * Imputação pela mediana, um método univariado, simples e robusto a outliers.
  * Imputação baseada em k-Nearest Neighbors (KNN), um método multivariado que estima os valores ausentes a partir de observações semelhantes no espaço das variáveis numéricas.

O objetivo é comparar o impacto dessas estratégias nas métricas finais do modelo, avaliando se abordagens mais complexas oferecem ganho real em desempenho preditivo neste problema.

A imputação de valores ausentes não é realizada nesta etapa para evitar vazamento de informação antes da separação dos dados. As estratégias de imputação serão incorporadas aos pipelines de pré-processamento e ajustadas exclusivamente no conjunto de treino durante a etapa de modelagem.

## Duplicates

In [ ]:
duplicated_rows = df.duplicated().sum()
print("Duplicated Rows:", duplicated_rows)

Não há linhas duplicadas no conjunto de dados.

## Outliers

In [ ]:
from preprocessing import count_outliers

continuous_vars = [
    "age",
    "avg_glucose_level",
    "bmi"
]

outliers = {column: count_outliers(df[column]) for column in continuous_vars}
print("Outliers:", outliers)

A coluna `avg_glucose_level` tem 627 outliers e a coluna `bmi` tem 119 outliers.

Existem diferentes estratégias para lidar com outliers, como a remoção das observações, *capping/winsorization*, a transformação das variáveis ou deixar os valores como estão.

Remover os outliers poderia resultar em perda significativa de informação.

Dessa forma, para as colunas `avg_glucose_level` e `bmi`, vamos aplicar uma transformação logarítmica, com o objetivo de reduzir a assimetria das distribuições e atenuar a influência de valores extremos.

In [ ]:
zero_or_negative_values = {column: (df[column] <= 0).sum() for column in continuous_vars}
print("Zero or Negative Values:", zero_or_negative_values)

Não existem valores zero ou negativos nessas colunas.

In [ ]:
df["avg_glucose_level"] = np.log1p(df["avg_glucose_level"])
df["bmi"] = np.log1p(df["bmi"])

In [ ]:
continuous_vars = [
    "age",
    "avg_glucose_level",
    "bmi"
]

outliers = {column: count_outliers(df[column]) for column in continuous_vars}
print("Outliers:", outliers)

A transformação logarítmica reduziu o número de outliers nas colunas `avg_glucose_level` e `bmi`.

In [ ]:
continuous_vars = [
    # "age",
    "avg_glucose_level",
    "bmi"
]

fig, axs = plt.subplots(
    nrows=2,
    ncols=2,
    gridspec_kw={"height_ratios": [2, 1]},
    figsize=(15, 6),
)

for i, var in enumerate(continuous_vars):
    sns.histplot(
        data=df,
        x=var,
        kde=True,
        ax=axs[0, i],
    )

    sns.boxplot(
        data=df,
        x=var,
        ax=axs[1, i],
    )

plt.tight_layout()
plt.show()

## Encoding

In [ ]:
categorical_vars = [
    "gender",
    "hypertension",
    "heart_disease",
    "ever_married",
    "work_type",
    "residence_type",
    "smoking_status",
]

In [ ]:
# check unique values in categorical columns
unique_categorical_values = {column: df[column].unique() for column in categorical_vars}
print(unique_categorical_values)

O valor `other` na coluna `gender` e `unknown` na coluna `smoking_status` não são esperados.

Dependendo do número de ocorrências dessas categorias, pode fazer sentido agrupá-las em uma das categorias existentes ou criar uma categoria separada para elas.

In [ ]:
other_gender_count = (df["gender"] == "other").sum()
print(other_gender_count)

In [ ]:
unknown_smoking_count = (df["smoking_status"] == "unknown").sum()
print(unknown_smoking_count)

Considerando que a categoria de gênero `other` possui apenas uma ocorrência, podemos remover essa linha, pois é improvável que ela tenha um impacto significativo na análise.

Já para a categoria unknown em `smoking_status`, como temos um número relativamente de ocorrências (1544), não é apropriado simplesmente remover essas linhas. Podemos tratar `unknown` como uma categoria separada.

In [ ]:
df = df[df["gender"] != "other"].copy()

In [ ]:
df.head()

In [ ]:
# map binary categorical variables to 0 and 1
df["ever_married"] = df["ever_married"].map({"no": 0, "yes": 1})

In [ ]:
df.to_csv("../data/processed/stroke_clean.csv", index=False)

# Modeling

In [ ]:
continuous_vars = [
    "age",
    "avg_glucose_level",
    "bmi"
]

binary_vars = [
    "hypertension",
    "heart_disease",
    "ever_married"
]

categorical_vars = [
    "gender",
    "work_type",
    "residence_type",
    "smoking_status",
]

In [ ]:
from preprocessing import build_preprocessor_median, build_preprocessor_knn

preprocessor_median = build_preprocessor_median(
    continuous_vars,
    categorical_vars,
    binary_vars,
)

preprocessor_knn = build_preprocessor_knn(
    continuous_vars,
    categorical_vars,
    binary_vars,
)

In [ ]:
preprocessor_median

In [ ]:
preprocessor_knn